# Lab 02 — Full tuning grid

Five predeclared groups cover all 13 fine-tune candidates for each architecture (26 trials total). Each saved version runs one group; combine all groups by validation before the final recommendation. AdamW/SGD, last block/last two blocks, separate head/backbone LR, cosine/fixed control. Batch 64/128, head 10 epochs, fine-tune min 10/max 30 with validation-loss patience 7. No Git push.

In [1]:
import torch
assert torch.cuda.is_available(), "Enable a CUDA GPU accelerator"
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), "count:", torch.cuda.device_count())

PyTorch: 2.11.0+cu128
GPU: Tesla T4 count: 2


## 1. Nạp source local

Cell bootstrap dưới đây chỉ giải nén source snapshot đã kiểm SHA256, không tải code từ GitHub.

In [2]:
import base64, hashlib, io, zipfile, os, sys, subprocess
from pathlib import Path
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '3e95748f789764f0793211c4d4b4f7493a3506c14cada22dc9f9043e25022fca'
ROOT = Path('/kaggle/working/lab02')
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
    bundle.extractall(ROOT)
os.chdir(ROOT)
print('Local source snapshot:', '3e95748f789764f0793211c4d4b4f7493a3506c14cada22dc9f9043e25022fca')


Local source snapshot: 3e95748f789764f0793211c4d4b4f7493a3506c14cada22dc9f9043e25022fca


In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorboard', 'markdown', 'pytest'], check=True)
print('Runtime ready; platform CUDA torch retained.')

Runtime ready; platform CUDA torch retained.


## 2. Dữ liệu và kiểm tra trước huấn luyện

Tải CIFAR-10/pretrained weights chính thức; dữ liệu và artifact ghi vào /kaggle/working. Test 1-batch và hợp đồng phải đạt trước khi train.

In [4]:
subprocess.run([sys.executable, '-u', 'scripts/download_assets.py', '--chunked'], check=True)
subprocess.run([sys.executable, '-u', 'scripts/analyze_data.py'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_contracts.py', 'tests/test_cached_prefix.py', 'tests/test_tuning.py', '-q'], check=True)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /kaggle/working/lab02/data/pretrained/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 161MB/s]
  0%|          | 0.00/30.8M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /kaggle/working/lab02/data/pretrained/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 169MB/s] 


Official assets ready.
/kaggle/working/lab02/outputs/eda
.........................                                                [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-pac

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', 'tests/test_contracts.py', 'tests/test_cached_prefix.py', 'tests/test_tuning.py', '-q'], returncode=0)

## 3. Huấn luyện tuning theo nhóm

Cell dưới chạy một nhóm được xác định trước từ config/tuned.json. Mỗi model giữ nguyên head search LR×batch và seed. Fine-tune reset về cùng selected head cho từng candidate. Bảng kết quả nhóm chỉ là kết quả tạm; phải gộp đủ 5 nhóm và chọn theo validation, không chọn bằng test.

In [5]:
import json
cfg = json.loads((ROOT / 'config/tuned.json').read_text())
all_candidates = cfg['finetune_candidates']
group = 0
cfg['finetune_candidates'] = all_candidates[group * 3:(group + 1) * 3]
assert cfg['finetune_candidates']
cfg['profile'] = f'cifar10_gpu_tuning_group_{group:02d}'
cfg['notes'] = f'Predeclared grid shard {group}/4; select globally by validation after all five shards; no test-based selection.'
config_path = ROOT / 'config' / f'tuning_group_{group:02d}.json'
config_path.write_text(json.dumps(cfg, indent=2))
print('GRID GROUP', group, [c['id'] for c in cfg['finetune_candidates']], flush=True)
subprocess.run([sys.executable, '-u', 'scripts/run.py', '--config', str(config_path), '--device', 'cuda'], check=True)

GRID GROUP 0 ['control', 'adamw_last_block_3e-06', 'adamw_last_block_1e-05']
Run: /kaggle/working/lab02/outputs/runs/20261027T042305Z_cifar10_gpu_tuning_group_00_c444e2bc
Device: cuda
resnet18 extract train: 64/45000
resnet18 extract train: 1664/45000
resnet18 extract train: 3264/45000
resnet18 extract train: 4864/45000
resnet18 extract train: 6464/45000
resnet18 extract train: 8064/45000
resnet18 extract train: 9664/45000
resnet18 extract train: 11264/45000
resnet18 extract train: 12864/45000
resnet18 extract train: 14464/45000
resnet18 extract train: 16064/45000
resnet18 extract train: 17664/45000
resnet18 extract train: 19264/45000
resnet18 extract train: 20864/45000
resnet18 extract train: 22464/45000
resnet18 extract train: 24064/45000
resnet18 extract train: 25664/45000
resnet18 extract train: 27264/45000
resnet18 extract train: 28864/45000
resnet18 extract train: 30464/45000
resnet18 extract train: 32064/45000
resnet18 extract train: 33664/45000
resnet18 extract train: 35264/450

CompletedProcess(args=['/usr/bin/python3', '-u', 'scripts/run.py', '--config', '/kaggle/working/lab02/config/tuning_group_00.json', '--device', 'cuda'], returncode=0)

## 4. Đóng gói bằng chứng

Source, logs, config, metrics, figures và checkpoint của run được giữ trong Kaggle Output. Không push Git. Kết quả chưa chạy không được ghi thành kết quả đã đạt.

In [6]:
import json, shutil
registry = json.loads((ROOT / 'outputs/selected_run.json').read_text())
RUN = Path(registry['run'])
subprocess.run([sys.executable, '-u', 'scripts/finalize.py', '--run', str(RUN)], check=True)
print('Completed grid group:', RUN)
print('Artifacts:', shutil.make_archive('/kaggle/working/lab02_tuning_artifacts', 'zip', ROOT / 'outputs'))

      model    stage  accuracy  macro_f1
   resnet18     head    0.8586  0.859006
   resnet18 finetune    0.9137  0.913663
densenet121     head    0.8790  0.879111
densenet121 finetune    0.9111  0.911231
Verified manifest: 266 files
HTML report: /kaggle/working/lab02/outputs/runs/20261027T042305Z_cifar10_gpu_tuning_group_00_c444e2bc/report_vi.html
Completed grid group: /kaggle/working/lab02/outputs/runs/20261027T042305Z_cifar10_gpu_tuning_group_00_c444e2bc
Artifacts: /kaggle/working/lab02_tuning_artifacts.zip
